In [ ]:
# Install nnsight
!pip install nnsight
!pip install --upgrade transformers torch
!pip install -U transformers
!pip install --upgrade google-cloud-translate

from IPython.display import clear_output
clear_output()

In [ ]:
import torch
import nnsight
from nnsight import LanguageModel
from google.colab import drive
import os
import pandas as pd
from google.cloud import translate
drive.mount('/content/drive')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Mounted at /content/drive


### vllm attempt

In [ ]:
############################################
# CONFIG
############################################

INPUT_FILE = "/content/drive/MyDrive/dataset_with_benign_column.csv"
OUTPUT_DIR = "/content/drive/Shareddrives/Jailbreaking Project/dataset"

MODEL_NAME = "google/translategemma-4b-it"

############################################
# LANGUAGE TIERS
############################################

TIER1_LANGS = {
    "en": "English",
    "es": "Spanish",
    "zh": "Chinese",
    "de": "German",
    "fr": "French"
}

TIER2_LANGS = {
    "ar": "Arabic",
    "ru": "Russian",
    "kr": "Korean",
    "jp": "Japanese"
}

TIER3_LANGS = {
    "tr": "Turkish",
    "id": "Indonesian",
    "hi": "Hindi",
    "sw": "Swahili"
}

TIER4_LANGS = {
    "yo": "Yoruba",
    "zu": "Zulu",
    "gd": "Scots Gaelic",
    "gn": "Guarani",
    "jv": "Javanese"
}

TIERS = {
    "tier_1": TIER1_LANGS,
    "tier_2": TIER2_LANGS,
    "tier_3": TIER3_LANGS,
    "tier_4": TIER4_LANGS
}

############################################
# LOAD MODEL
############################################

print("Loading model...")

llm = LLM(model=MODEL_NAME)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=256
)

############################################
# TRANSLATION FUNCTION
############################################

def translate(text, source_lang, target_lang):

    prompt = f"""Translate the following text from {source_lang} to {target_lang}.
Only output the translation.

Text: {text}
Translation:"""

    outputs = llm.generate([prompt], sampling_params)

    translation = outputs[0].outputs[0].text.strip()

    return translation

############################################
# STANDARD TRANSLATION
############################################

def standard_translation(texts, lang_name, lang_code):

    translated = []

    for text in texts:

        if lang_code == "en":
            translated.append(text)

        else:
            translated.append(
                translate(text, "English", lang_name)
            )

    return translated


############################################
# TRANSLATIONESE
############################################

def translationese(texts, lang_name, lang_code):

    translated = []

    for text in texts:

        if lang_code == "en":
            translated.append(text)
            continue

        # English → Target
        step1 = translate(text, "English", lang_name)

        # Target → English
        step2 = translate(step1, lang_name, "English")

        # English → Target
        step3 = translate(step2, "English", lang_name)

        translated.append(step3)

    return translated


############################################
# SAVE CSV
############################################

def save_prompts(prompts, path):

    df = pd.DataFrame({"prompt": prompts})
    df.to_csv(path, index=False)


############################################
# DATASET GENERATION
############################################

def generate_dataset(df, mode):

    harmful = df["prompt"].tolist()
    harmless = df["prompt_harmless"].tolist()

    for tier_name, langs in TIERS.items():

        for lang_code, lang_name in langs.items():

            print(f"{mode} | {tier_name} | {lang_code}")

            lang_dir = os.path.join(
                OUTPUT_DIR,
                mode,
                tier_name,
                lang_code
            )

            os.makedirs(lang_dir, exist_ok=True)

            if mode == "standard":

                harmful_out = standard_translation(
                    harmful, lang_name, lang_code
                )

                harmless_out = standard_translation(
                    harmless, lang_name, lang_code
                )

            else:

                harmful_out = translationese(
                    harmful, lang_name, lang_code
                )

                harmless_out = translationese(
                    harmless, lang_name, lang_code
                )

            save_prompts(
                harmful_out,
                os.path.join(lang_dir, "harmful_prompts.csv")
            )

            save_prompts(
                harmless_out,
                os.path.join(lang_dir, "harmless_prompts.csv")
            )


############################################
# MAIN
############################################

def main():

    print("Loading dataset...")
    df = pd.read_csv(INPUT_FILE)

    print("Generating STANDARD translations")
    generate_dataset(df, "standard")

    print("Generating TRANSLATIONESE translations")
    generate_dataset(df, "translationese")


if __name__ == "__main__":
    main()

Loading model...
INFO 03-09 06:38:40 [utils.py:238] non-default args: {'disable_log_stats': True, 'model': 'google/translategemma-4b-it'}


ValidationError: 1 validation error for ModelConfig
  Value error, rope_parameters should have a 'rope_type' key [type=value_error, input_value=ArgsKwargs((), {'model': ...rocessor_plugin': None}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

### nnsight/huggingface attempt

In [ ]:
############################################
# CONFIG
############################################

INPUT_FILE = "/content/drive/MyDrive/dataset_with_benign_column.csv"
OUTPUT_DIR = "/content/drive/Shareddrives/Jailbreaking Project/dataset"

############################################
# LANGUAGE TIERS
############################################

TIER1_LANGS = {
    "en": "English",
    "es": "Spanish",
    "zh": "Chinese",
    "de": "German",
    "fr": "French"
}

TIER2_LANGS = {
    "ar": "Arabic",
    "ru": "Russian",
    "kr": "Korean",
    "jp": "Japanese"
}

TIER3_LANGS = {
    "tr": "Turkish",
    "id": "Indonesian",
    "hi": "Hindi",
    "sw": "Swahili"
}

TIER4_LANGS = {
    "yo": "Yoruba",
    "zu": "Zulu",
    "gd": "Scots Gaelic",
    "gn": "Guarani",
    "jv": "Javanese"
}

TIERS = {
    "tier_1": TIER1_LANGS,
    "tier_2": TIER2_LANGS,
    "tier_3": TIER3_LANGS,
    "tier_4": TIER4_LANGS
}

############################################
# LOAD MODEL
############################################

translation_llm = LanguageModel(
    "google/translategemma-4b-it",
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    dispatch=True
)

tokenizer = translation_llm.tokenizer

In [ ]:
############################################
# SINGLE TRANSLATION FUNCTION
############################################

def translate(text, source_lang, target_lang):

    prompt = f"""Translate the following text from {source_lang} to {target_lang}.
Only output the translation.

Text: {text}
Translation:"""

    with translation_llm.generate(max_new_tokens=256, do_sample=False) as tracer:

        with tracer.invoke(prompt):
            output = translation_llm.generator.output.save()

    decoded = tokenizer.decode(output[0],skip_special_tokens=True).strip()

    return decoded

############################################
# STANDARD TRANSLATION
############################################

def standard_translation(texts, lang_name, lang_code):

    translated = []

    for text in texts:

        if lang_code == "en":
            translated.append(text)
        else:
            translated.append(
                translate(text, "English", lang_name)
            )

    return translated


############################################
# TRANSLATIONESE
############################################

def translationese(texts, lang_name, lang_code):

    translated = []

    for text in texts:

        if lang_code == "en":
            translated.append(text)
            continue

        # English → Target
        step1 = translate(text, "English", lang_name)

        # Target → English
        step2 = translate(step1, lang_name, "English")

        # English → Target
        step3 = translate(step2, "English", lang_name)

        translated.append(step3)

    return translated


############################################
# SAVE CSV
############################################

def save_prompts(prompts, path):

    df = pd.DataFrame({"prompt": prompts})
    df.to_csv(path, index=False)


############################################
# DATASET GENERATION
############################################

def generate_dataset(df, mode):

    harmful = df["prompt"].tolist()
    harmless = df["prompt_harmless"].tolist()

    for tier_name, langs in TIERS.items():

        for lang_code, lang_name in langs.items():

            print(f"{mode} | {tier_name} | {lang_code}")

            lang_dir = os.path.join(
                OUTPUT_DIR,
                mode,
                tier_name,
                lang_code
            )

            os.makedirs(lang_dir, exist_ok=True)

            if mode == "standard":

                harmful_out = standard_translation(
                    harmful, lang_name, lang_code
                )

                harmless_out = standard_translation(
                    harmless, lang_name, lang_code
                )

            else:

                harmful_out = translationese(
                    harmful, lang_name, lang_code
                )

                harmless_out = translationese(
                    harmless, lang_name, lang_code
                )

            save_prompts(
                harmful_out,
                os.path.join(lang_dir, "harmful_prompts.csv")
            )

            save_prompts(
                harmless_out,
                os.path.join(lang_dir, "harmless_prompts.csv")
            )


############################################
# MAIN
############################################

def main():

    print("Loading dataset...")
    df = pd.read_csv(INPUT_FILE)

    print("Generating STANDARD dataset...")
    generate_dataset(df, "standard")

    print("Generating TRANSLATIONESE dataset...")
    generate_dataset(df, "translationese")


if __name__ == "__main__":
    main()

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Loading dataset...
Generating STANDARD dataset...
standard | tier_1 | en
standard | tier_1 | es


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for o

KeyboardInterrupt: 

### Google translate API attempt

In [ ]:
from google.colab import userdata
import json
import os

key_json = userdata.get("GOOGLE_SERVICE_ACCOUNT")

with open("/content/service_account.json", "w") as f:
    f.write(key_json)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/service_account.json"

print("Authentication loaded from secret.")

Authentication loaded from secret.


In [ ]:
############################################
# IMPORTS
############################################

import os
import pandas as pd
from google.cloud import translate_v3 as translate

############################################
# CONFIG
############################################

PROJECT_ID = "colab-translate-489801"
LOCATION = "global"

INPUT_FILE = "/content/drive/MyDrive/dataset_with_benign_column.csv"
OUTPUT_DIR = "/content/drive/MyDrive/dataset"

client = translate.TranslationServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

# Google Cloud Translation API limit for total codepoints per request
MAX_CODEPOINTS_PER_REQUEST = 30000 # Slightly less than 30720 for buffer

############################################
# LANGUAGE TIERS
############################################

TIER1_LANGS = {
    "en": "English",
    "es": "Spanish",
    "zh": "Chinese",
    "de": "German",
    "fr": "French"
}

TIER2_LANGS = {
    "ar": "Arabic",
    "ru": "Russian",
    "ko": "Korean",
    "ja": "Japanese"
}

TIER3_LANGS = {
    "tr": "Turkish",
    "id": "Indonesian",
    "hi": "Hindi",
    "sw": "Swahili"
}

TIER4_LANGS = {
    "yo": "Yoruba",
    "zu": "Zulu",
    "gd": "Scots Gaelic",
    "gn": "Guarani",
    "jv": "Javanese"
}

TIERS = {
    "tier_1": TIER1_LANGS,
    "tier_2": TIER2_LANGS,
    "tier_3": TIER3_LANGS,
    "tier_4": TIER4_LANGS
}

############################################
# BATCH TRANSLATION FUNCTION
############################################

def translate_batch(texts, target_lang, source_lang="en"):

    if target_lang == "en":
        return texts

    translated_texts = []
    current_batch = []
    current_batch_codepoints = 0

    for text in texts:
        text_codepoints = len(text)

        # If adding the next text exceeds the limit, translate the current_batch
        if current_batch_codepoints + text_codepoints > MAX_CODEPOINTS_PER_REQUEST and current_batch:
            response = client.translate_text(
                contents=current_batch,
                target_language_code=target_lang,
                source_language_code=source_lang,
                parent=parent,
            )
            translated_texts.extend([t.translated_text for t in response.translations])
            current_batch = []
            current_batch_codepoints = 0

        current_batch.append(text)
        current_batch_codepoints += text_codepoints

    # Translate any remaining texts in current_batch
    if current_batch:
        response = client.translate_text(
            contents=current_batch,
            target_language_code=target_lang,
            source_language_code=source_lang,
            parent=parent,
        )
        translated_texts.extend([t.translated_text for t in response.translations])

    return translated_texts

############################################
# STANDARD TRANSLATION
############################################

def standard_translation(texts, lang_code):

    return translate_batch(texts, lang_code)

############################################
# TRANSLATIONESE
############################################

def translationese(texts, lang_code):

    if lang_code == "en":
        return texts

    # English → Target
    step1 = translate_batch(texts, lang_code)

    # Target → English
    step2 = translate_batch(step1, "en", source_lang=lang_code)

    # English → Target
    step3 = translate_batch(step2, lang_code)

    return step3


############################################
# CODE SWITCHING
############################################

import re

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)


def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return text


def code_switch_pair(harmful_text, harmless_text, lang_code):

    if lang_code == "en":
        return harmful_text

    harmful_tokens = tokenize(harmful_text)
    harmless_tokens = tokenize(harmless_text)

    tokens_to_translate = []
    token_indices = []

    min_len = min(len(harmful_tokens), len(harmless_tokens))

    for i in range(min_len):

        h_tok = harmful_tokens[i]
        b_tok = harmless_tokens[i]

        # only translate if tokens differ AND are words
        if h_tok.lower() != b_tok.lower() and h_tok.isalpha():

            tokens_to_translate.append(h_tok)
            token_indices.append(i)

    if len(tokens_to_translate) == 0:
        return harmful_text

    translated = translate_batch(tokens_to_translate, lang_code)

    for idx, new_tok in zip(token_indices, translated):
        harmful_tokens[idx] = new_tok

    return detokenize(harmful_tokens)

def code_switching(harmful_texts, harmless_texts, lang_code):

    switched = []

    for h, b in zip(harmful_texts, harmless_texts):

        switched.append(code_switch_pair(h, b, lang_code))

    return switched

############################################
# SAVE CSV
############################################

def save_prompts(prompts, path):

    df = pd.DataFrame({"prompt": prompts})
    df.to_csv(path, index=False)

############################################
# DATASET GENERATION
############################################

def generate_dataset(df, mode):

    harmful = df["prompt"].tolist()
    harmless = df["prompt_harmless"].tolist()

    for tier_name, langs in TIERS.items():

        for lang_code, lang_name in langs.items():

            print(f"{mode} | {tier_name} | {lang_code}")

            lang_dir = os.path.join(
                OUTPUT_DIR,
                mode,
                tier_name,
                lang_code
            )

            os.makedirs(lang_dir, exist_ok=True)

            if mode == "standard":
                harmful_out = standard_translation(harmful, lang_code)
                harmless_out = standard_translation(harmless, lang_code)

            elif mode == "translationese":
                harmful_out = translationese(harmful, lang_code)
                harmless_out = translationese(harmless, lang_code)

            else:
                harmful_out = code_switching(harmful, harmless, lang_code)


            save_prompts(harmful_out,os.path.join(lang_dir, "harmful_prompts.csv"))
            if mode != "code_switching":
              save_prompts(harmless_out,os.path.join(lang_dir, "harmless_prompts.csv"))

############################################
# MAIN
############################################

def main():

    print("Loading dataset...")
    df = pd.read_csv(INPUT_FILE)

    # print("Generating STANDARD dataset...")
    # generate_dataset(df, "standard")

    # print("Generating TRANSLATIONESE dataset...")
    # generate_dataset(df, "translationese")


    print("Generating CODESWITCHING dataset...")
    generate_dataset(df, "code_switching")

if __name__ == "__main__":
    main()

Loading dataset...
Generating CODESWITCHING dataset...
code_switching | tier_1 | en
code_switching | tier_1 | es
code_switching | tier_1 | zh
code_switching | tier_1 | de
code_switching | tier_1 | fr
code_switching | tier_2 | ar
code_switching | tier_2 | ru
code_switching | tier_2 | ko
code_switching | tier_2 | ja
code_switching | tier_3 | tr
code_switching | tier_3 | id
code_switching | tier_3 | hi
code_switching | tier_3 | sw
code_switching | tier_4 | yo
code_switching | tier_4 | zu
code_switching | tier_4 | gd
code_switching | tier_4 | gn
code_switching | tier_4 | jv


In [ ]:
# Load the dataset again
import pandas as pd
df = pd.read_csv(INPUT_FILE)

# Temporarily redefine TIERS to only include tier_4 for this run
# Make a copy to avoid modifying the original dictionary defined in a previous cell
original_tiers_copy = TIERS.copy()

TIERS = {
    "tier_4": {
        # "yo": "Yoruba",
        # "zu": "Zulu",
        "gd": "Scots Gaelic",
        "gn": "Guarani",
        "jv": "Javanese"
    }
}

print("Generating CODESWITCHING dataset...")
generate_dataset(df, "code_switching")

# Restore the original TIERS dictionary if needed for subsequent operations
TIERS = original_tiers_copy
print("Dataset generation for TIER 4 (translationese) complete.")
print("Original TIERS dictionary restored.")

Generating CODESWITCHING dataset...
code_switching | tier_4 | gd
code_switching | tier_4 | gn
code_switching | tier_4 | jv
Dataset generation for TIER 4 (translationese) complete.
Original TIERS dictionary restored.
